# 03 - Pretrained Model (Chronos Zero-Shot)

Mục tiêu: chạy pretrained time-series model (Chronos) ở chế độ
zero-shot trên đúng tập series đã dùng ở 02_baseline_gb.ipynb, để
đảm bảo so sánh công bằng giữa GB baseline và pretrained model.

In [1]:
import sys
sys.path.append('../src')

import time
import pandas as pd
import numpy as np

from data_loader import load_and_preprocess_m4_monthly
from models_pretrained import (
    load_pretrained_model, predict_series, predict_batch, build_series_dict
)
from evaluation import calculate_metrics

/home/wotttoo/Desktop/AIO2026/Module 3/Conquer/hybrid-forecasting-gb-pretrained/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
assert os.path.exists("../data/processed/sampled_metadata.csv"), \
    "Chưa có sampled_metadata.csv — cần chạy length_grouping.py (ở 01_eda.ipynb) trước"

In [3]:
# Dùng ĐÚNG tập series đã sample ở 01_eda.ipynb, để so sánh công bằng
# với kết quả GB baseline ở 02_baseline_gb.ipynb
sampled_metadata = pd.read_csv("../data/processed/sampled_metadata.csv")
selected_ids = sampled_metadata["unique_id"].tolist()

# Lấy dữ liệu THÔ (chưa qua feature engineering) — Chronos không cần
# lag/rolling/date features như GB, chỉ cần chuỗi giá trị gốc
train_df, test_df = load_and_preprocess_m4_monthly(
    "../data/raw/Train/Monthly-train.csv",
    "../data/raw/Test/Monthly-test.csv",
    series_ids=selected_ids
)

print(f"Số series sẽ chạy: {train_df['unique_id'].nunique()}")

--- Bước 1 & 2: Đọc và Melt dữ liệu ---
--- Bước 3: Tái tạo mốc thời gian ---
Hoàn thành! Kích thước Train: (90824, 3), Test: (8334, 3)
Số series sẽ chạy: 463


In [4]:
from features import make_train_test_split

# Chia giống HỆT cách 02_baseline_gb.ipynb đã làm
train_split, val_split = make_train_test_split(train_df, test_horizon=18)

# series_dict giờ chỉ chứa phần lịch sử TRƯỚC 18 tháng cuối
# (Chronos sẽ dự báo đúng phần đã bị giấu này, giống hệt GB)
series_dict = build_series_dict(train_split)

# Kiểm tra nhanh 1 series mẫu
sample_uid = list(series_dict.keys())[0]
print(f"Ví dụ series {sample_uid}: {len(series_dict[sample_uid])} điểm dữ liệu")
print(series_dict[sample_uid][:5])

Ví dụ series M10200: 94 điểm dữ liệu
[1154. 1159. 1165. 1155. 1154.]


In [5]:
# Dùng bản "small" — cân bằng giữa tốc độ và độ chính xác, chạy được trên CPU
pipeline = load_pretrained_model("amazon/chronos-t5-small", device="cpu")

Đang tải model amazon/chronos-t5-small lên cpu...


`torch_dtype` is deprecated! Use `dtype` instead!


Tải model thành công.


## Test nhanh trên vài series trước khi chạy full

Chronos khá tốn thời gian — nên ước lượng tốc độ trước khi chạy
toàn bộ ~500 series, tránh chạy full rồi mới phát hiện quá chậm.

In [6]:
test_ids = list(series_dict.keys())[:5]
test_dict = {uid: series_dict[uid] for uid in test_ids}

start_time = time.time()
test_result = predict_batch(pipeline, test_dict, horizon=18, num_samples=20)
elapsed = time.time() - start_time

avg_time_per_series = elapsed / len(test_ids)
estimated_total = avg_time_per_series * len(series_dict)

print(f"Thời gian chạy 5 series: {elapsed:.1f}s")
print(f"Trung bình mỗi series: {avg_time_per_series:.2f}s")
print(f"Ước tính thời gian chạy toàn bộ {len(series_dict)} series: "
      f"{estimated_total/60:.1f} phút")

Chronos zero-shot: 100%|██████████| 5/5 [00:02<00:00,  1.77it/s]

Thời gian chạy 5 series: 2.8s
Trung bình mỗi series: 0.56s
Ước tính thời gian chạy toàn bộ 463 series: 4.4 phút


## Chạy full inference

Nếu thời gian ước tính ở trên quá lâu (>30-40 phút), cân nhắc:
- Giảm `num_samples` (VD: 10 thay vì 20) — giảm độ chính xác nhẹ,
  tăng tốc đáng kể
- Dùng bản model nhỏ hơn: "amazon/chronos-t5-tiny"
- Giảm số lượng series trong sampled_metadata.csv (sample_size nhỏ hơn)

In [7]:
start_time = time.time()

predictions_long = predict_batch(
    pipeline, series_dict, horizon=18, num_samples=20, show_progress=True
)

elapsed = time.time() - start_time
print(f"\nHoàn thành trong {elapsed/60:.1f} phút")
print(f"Số dòng dự báo: {len(predictions_long)}")

Chronos zero-shot: 100%|██████████| 463/463 [03:48<00:00,  2.03it/s]


Hoàn thành trong 3.8 phút
Số dòng dự báo: 8334


In [8]:
val_split_sorted = val_split.sort_values(["unique_id", "ds"]).copy()
val_split_sorted["step"] = val_split_sorted.groupby("unique_id").cumcount() + 1

predictions_merged = predictions_long.merge(
    val_split_sorted[["unique_id", "step", "ds", "y"]],
    on=["unique_id", "step"], how="left"
)

In [9]:
predictions_merged = predictions_merged.merge(
    sampled_metadata[["unique_id", "category", "length_group"]],
    on="unique_id", how="left"
)

# Kiểm tra không mất series nào sau các bước merge
n_missing = predictions_merged["length_group"].isna().sum()
if n_missing > 0:
    print(f"⚠️ Cảnh báo: {n_missing} dòng thiếu length_group sau merge")
else:
    print("Merge thành công, không thiếu length_group.")

Merge thành công, không thiếu length_group.


In [10]:
metrics = calculate_metrics(
    predictions_merged["y"], predictions_merged["pred_pretrained"]
)
print("Chronos (zero-shot) Metrics:", metrics)

Chronos (zero-shot) Metrics: {'MAE': 565.1114432883119, 'RMSE': np.float64(1573.7498013895918), 'MAPE': np.float64(12.707121169977695)}


In [11]:
results_df = predictions_merged[
    ["unique_id", "length_group", "category", "ds", "y", "pred_pretrained"]
].copy()

results_df.to_csv("../outputs/metrics/pretrained_predictions.csv", index=False)
print(f"Đã lưu {len(results_df)} dòng vào pretrained_predictions.csv")

Đã lưu 8334 dòng vào pretrained_predictions.csv


## Kết luận

Đã lưu `pretrained_predictions.csv` với cùng cấu trúc
(`unique_id, length_group, category, ds, y`) như
`gb_baseline_predictions.csv` từ 02_baseline_gb.ipynb — sẵn sàng cho
bước ensemble ở 04_ensemble.ipynb, chỉ cần merge 2 file theo
(`unique_id`, `ds`).